In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
load_dotenv()
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

### Document Loading

In [3]:
doc_loader=PyPDFLoader(file_path=r"D:\Projects\RAG_in_pdf\pdfs\llm_paper.pdf")

In [4]:
documents = doc_loader.load()


### Chunking


In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 317


In [27]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2273.20it/s]


### Storing the chunks to the Vector Storage

In [28]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [33]:
vector_store.save_local("faiss_index")

### Performing Similarity Search

In [31]:
query = "What is GPT-3?"

result = vector_store.similarity_search_with_score(
    query,
    k=10
)

for doc, score in result:
    print(doc.page_content)
    print("Score:", score)
    print("-" * 50)

context = "\n\n".join(
    doc.page_content for doc, score in result
)

more systematic way, we also evaluate GPT-3 on a standardized collection of datasets, the SuperGLUE benchmark
[WPN+19] [WPN+19] [CLC+19] [DMST19] [RBG11] [KCR+18] [ZLL+18] [DGM06] [BHDD+06] [GMDD07]
[BDD+09] [PCC18] [PHR+18]. GPT-3’s test-set performance on the SuperGLUE dataset is shown in Table 3.8. In the
few-shot setting, we used 32 examples for all tasks, sampled randomly from the training set. For all tasks except WSC
18
Score: 0.9065987
--------------------------------------------------
predecessor GPT-2, it still has notable weaknesses in text synthesis and several NLP tasks. On text synthesis, although
the overall quality is high, GPT-3 samples still sometimes repeat themselves semantically at the document level, start to
lose coherence over sufﬁciently long passages, contradict themselves, and occasionally contain non-sequitur sentences
or paragraphs. We will release a collection of 500 uncurated unconditional samples to help provide a better sense of
GPT-3’s limitations and 

In [32]:
prompt = f"""
Answer the question based only on the context provided below.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I don't know based on the provided document."
"""

response = llm.invoke(prompt)

print(response.text)

Based on the provided context, **GPT-3** is a pre-trained language model evaluated across a broad spectrum of tasks (including text synthesis, discrete language tasks, the SuperGLUE benchmark, translation, and arithmetic) using zero-, one-, and few-shot in-context learning. 

Key details provided about GPT-3 include:
* **Scale:** Its dataset and model size are about two orders of magnitude larger than its predecessor (GPT-2), and its dataset includes a large amount of Common Crawl data. One mentioned variant is "GPT-3 175B."
* **Capabilities:** It displays strong proficiency on tasks like small-digit arithmetic and meta-learning across zero-, one-, and few-shot settings.
* **Limitations:** It has weaknesses in text synthesis (e.g., occasional semantic repetition, loss of coherence over long passages, self-contradictions) and difficulty with "common sense physics" questions.
